In [1]:
!pip install pandas

**Cargar datos desde GitHub**

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/kevinnumanzor17/etl-seguros-pipeline/refs/heads/main/data/raw/corredores.csv"

df = pd.read_csv(url)

df.head()

,id_corredor,nombre,zona,nivel,anios_experiencia
0,1,José López Flores,Paracentral,Mid,4.0
1,2,José Ortiz García,Centro,Junior,0.0
2,3,María Ramírez Cruz,Centro,SENIOR,6.0
3,4,Fernanda Rojas Cruz,NaN,SENIOR,8.0
4,5,Ana Gómez Rojas,NaN,Senior,4.0


**Exploración**

In [3]:
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_corredor        80 non-null     int64  
 1   nombre             80 non-null     object 
 2   zona               63 non-null     object 
 3   nivel              68 non-null     object 
 4   anios_experiencia  76 non-null     float64
dtypes: float64(1), int64(1), object(3)
memory usage: 3.3+ KB


,0
id_corredor,0
nombre,0
zona,17
nivel,12
anios_experiencia,4


**Limpieza de datos**

In [4]:
corredores = df.copy()

# limpiar nombres de columnas
corredores.columns = corredores.columns.str.strip().str.lower()

# limpiar espacios en texto
for col in corredores.select_dtypes(include='object').columns:
    corredores[col] = corredores[col].astype(str).str.strip()

# convertir vacíos en null
corredores = corredores.replace(r'^\s*$', pd.NA, regex=True)

# eliminar duplicados
corredores = corredores.drop_duplicates()

**convertir anios_experiencia a número**

In [5]:
corredores['anios_experiencia'] = pd.to_numeric(
    corredores['anios_experiencia'],
    errors='coerce'
)

**Separar válidos y rechazados**

In [6]:
validos = corredores[
    corredores['nombre'].notna() &
    corredores['zona'].notna() &
    corredores['anios_experiencia'].notna()
].copy()

rechazados = corredores[
    corredores['nombre'].isna() |
    corredores['zona'].isna() |
    corredores['anios_experiencia'].isna()
].copy()

**Motivo de rechazo**

In [7]:
def motivo(row):
    motivos = []

    if pd.isna(row['nombre']):
        motivos.append("nombre_vacio")

    if pd.isna(row['zona']):
        motivos.append("zona_vacio")

    if pd.isna(row['anios_experiencia']):
        motivos.append("experiencia_invalida")

    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

**Exportar archivos**

In [8]:
validos.to_csv("corredores_curated.csv", index=False)
rechazados.to_csv("corredores_rejects.csv", index=False)